# Resolve Wikipedia references

Resolve Wikipedia reference links for the pathogen records selected in the saved V4 run. The association table is keyed by `pathogen_id`, preserving stable V4 taxonomy identity. Lookups checkpoint to CSV after each API batch and resume completed records on rerun. Only exact Wikipedia article resolution or exact GBIF species matches receive links.

Run `00_workflow` and `02_runs_and_pubmed` first. The selected run is read from `data/v4/runs/<run-id>/resolved_config.json`; when the run selected pathogen groups, the full resolved pathogen selection is included. Set `FORCE_REFRESH = True` to repeat lookups for records in scope.

## Step 1 · Load the resolved run and taxonomy


In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the Pathogen Atlas V4 repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('08_wikipedia_references', RUN_ID)
RUN_PATH = ROOT / 'data/v4/runs' / RUN_ID / 'resolved_config.json'
if not RUN_PATH.exists():
    raise FileNotFoundError(f'Run notebook 02 first; resolved selection is missing: {RUN_PATH}')
resolved_run = json.loads(RUN_PATH.read_text(encoding='utf-8'))
pathogen_ids = set(map(str, resolved_run['pathogen_ids']))
TAXONOMY_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
PATHOGEN_TABLE = TAXONOMY_ROOT / 'pathogens.parquet'
pathogens = pd.read_parquet(PATHOGEN_TABLE, columns=['pathogen_id','pathogen_group','accepted_name','rank'])
scope = pathogens.loc[pathogens.pathogen_id.astype(str).isin(pathogen_ids) & pathogens['rank'].astype(str).str.casefold().eq('species')]
PATHOGEN_GROUPS = sorted(scope.pathogen_group.astype(str).unique())
OUTPUT_PATH = ROOT / 'data/v4/references/release/08_wikipedia_references.ipynb'
FORCE_REFRESH = False
existing = pd.read_csv(OUTPUT_PATH, keep_default_na=False) if OUTPUT_PATH.exists() else pd.DataFrame()
display(Markdown(f"**Run:** `{RUN_ID}` · **Groups:** {', '.join(PATHOGEN_GROUPS)} · **Species records:** {scope.pathogen_id.nunique():,} · **Existing rows:** {len(existing):,} · **Force refresh:** {FORCE_REFRESH}"))
LOG.info('Reference lookup prepared: run=%s groups=%s species=%d existing=%d', RUN_ID, PATHOGEN_GROUPS, scope.pathogen_id.nunique(), len(existing))


## Step 2 · Resolve and save Wikipedia associations

The lookup writes one row per pathogen ID, saves progress after every request batch, and reuses completed rows on rerun.

In [ ]:
from pathogen_atlas_v4.references.links import resolve_wikipedia_links
service = 'Wikipedia'

links = resolve_wikipedia_links(PATHOGEN_TABLE, OUTPUT_PATH, PATHOGEN_GROUPS, force_refresh=FORCE_REFRESH, pathogen_ids=pathogen_ids)
found = links.match_status.eq('found')
display(Markdown(f"**Saved:** `{OUTPUT_PATH.relative_to(ROOT)}` · **Records:** {len(links):,} · **{service} references found:** {int(found.sum()):,} · **No exact reference:** {int((~found).sum()):,}"))
display(links.head(12))
LOG.info('Reference lookup complete: service=Wikipedia records=%d found=%d output=%s', len(links), int(found.sum()), OUTPUT_PATH)
